# Fase 3: Entrenamiento de Modelos Tradicionales y BETO

Este notebook realiza el entrenamiento del clasificador híbrido BETO + Características Manuales. También entrena clasificadores tradicionales (SVM, Random Forest, XGBoost, MLP, Voting Classifier) sobre las mismas características como base de comparación.

In [ ]:
# 1. Conexión con Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Instalación de dependencias (para Colab con GPU)
!pip install transformers imbalanced-learn xgboost scikit-learn joblib tqdm

In [ ]:
# 3. Carga de librerías y verificación de GPU
import os
import gc
import joblib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizerFast, AutoModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo utilizado:", device)
if device.type == 'cuda':
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Cargar dataset seleccionado
PATH = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"
df = pd.read_json(PATH + "df_train2_featselect2.jsonl", orient='records', lines=True)
print(f"Dataset cargado. Forma: {df.shape}")

### Parte A: Clasificadores Tradicionales (Voting Classifier)

In [ ]:
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer

# 1. Definir columnas de features
selected_features = [
    'MeanWordLen', 'LexicalDiversity', 'MeanSentenceLen', 'StdevSentenceLen',
    'DocumentLen', 'WordsPerText', 'SentencesPerText', 'num_words', 'num_chars',
    'irony_score', 'prop_NOUN', 'prop_VERB', 'prop_ADJ', 'rhetorical_questions',
    'avg_depth', 'Flesch Score', 'Lexical Entropy', 'Syntactic Repetition', 'Unusual Word Frequency'
]

# 2. Vectorizador TF-IDF sobre el texto preprocesado
print("Ajustando TF-IDF...")
vectorizer = TfidfVectorizer(max_features=3000)
tfidf_matrices = vectorizer.fit_transform(df['transcription_processed']).toarray()

# 3. Combinar TF-IDF con features manuales
manual_features = df[selected_features].values
X_combined = np.concatenate([tfidf_matrices, manual_features], axis=1)
y = df['label'].values

# Imputar valores nulos (si hay alguno)
imputer = SimpleImputer(strategy='mean')
X_combined = imputer.fit_transform(X_combined)

# Normalizar características combinadas
scaler = MinMaxScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

# 4. Partición Train / Test
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    X_combined_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Balancear clases con SMOTE
print("Aplicando SMOTE...")
smote = SMOTE(random_state=42)
X_train_ml_res, y_train_ml_res = smote.fit_resample(X_train_ml, y_train_ml)

# 5. Configurar y entrenar Voting Classifier
clf1 = SVC(probability=True, kernel='linear', random_state=42)
clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf3 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
clf4 = MLPClassifier(max_iter=500, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('svc', clf1), ('rf', clf2), ('xgb', clf3), ('mlp', clf4)],
    voting='soft'
)

print("Entrenando Voting Classifier...")
voting_clf.fit(X_train_ml_res, y_train_ml_res)

# Evaluar tradicional
preds_ml = voting_clf.predict(X_test_ml)
print("\n--- reporte Clasificadores Tradicionales (Voting) ---")
print(classification_report(y_test_ml, preds_ml))

# Guardar vectorizador y scaler para producción
joblib.dump(vectorizer, PATH + "tfidf_vectorizer.pkl")
joblib.dump(scaler, PATH + "minmax_scaler.pkl")
print("Vectorizador y Scaler guardados en Drive.")

### Parte B: Clasificador Híbrido BETO + Características

In [ ]:
# 1. Inicializar Tokenizer y Modelo Base de BETO
BETO_MODEL_NAME = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = BertTokenizerFast.from_pretrained(BETO_MODEL_NAME)
beto_base = AutoModel.from_pretrained(BETO_MODEL_NAME)

# Guardar el tokenizer
tokenizer_dir = PATH + "tokenizer_files/"
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)
print(f"Tokenizer guardado en: {tokenizer_dir}")

In [ ]:
# 2. Definir Arquitectura de Clasificador Híbrido
class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_extra_features):
        super().__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(bert_model.config.hidden_size + num_extra_features, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 2)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_ids, attention_mask, extra_features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = torch.cat((cls_output, extra_features), dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.softmax(self.fc2(x))

In [ ]:
# 3. Preparación de Datasets de Tensores para PyTorch
# Tokenización del texto original
transcriptions = df['transcription'].tolist()
encoded_data = tokenizer.batch_encode_plus(
    transcriptions,
    add_special_tokens=True,
    return_attention_mask=True,
    padding=True,
    max_length=64,
    truncation=True,
    return_tensors='pt'
)

input_ids = encoded_data['input_ids']
attention_masks = encoded_data['attention_mask']

# Normalizar las características adicionales
# Usamos las mismas que pasaron por MinMaxScaler antes
extra_features = torch.tensor(X_combined_scaled, dtype=torch.float32)
labels = torch.tensor(df['label'].values, dtype=torch.long)

# Dividir índices para train/val (80/20)
train_idx, val_idx = train_test_split(
    np.arange(len(labels)), test_size=0.2, random_state=42, stratify=labels
)

# Dataloaders
batch_size = 16

train_dataset = TensorDataset(input_ids[train_idx], attention_masks[train_idx], extra_features[train_idx], labels[train_idx])
val_dataset = TensorDataset(input_ids[val_idx], attention_masks[val_idx], extra_features[val_idx], labels[val_idx])

train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=batch_size)

In [ ]:
# 4. Inicializar Clasificador Híbrido, Optimizador y Scheduler
num_extra_features = X_combined_scaled.shape[1]  # 3019 (3000 TF-IDF + 19 manuales)
model = BertClassifier(beto_base, num_extra_features).to(device)

epochs = 4
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = nn.NLLLoss()

In [ ]:
# 5. Ciclo de Entrenamiento de la Red Neuronal
best_val_loss = float('inf')

for epoch in range(epochs):
    print(f"\n====== Epoca {epoch+1} / {epochs} ======")
    model.train()
    total_train_loss = 0
    
    for batch in tqdm(train_dataloader, desc="Entrenando"):
        b_input_ids = batch[0].to(device)
        b_attn_mask = batch[1].to(device)
        b_features = batch[2].to(device)
        b_labels = batch[3].to(device)
        
        model.zero_grad()
        outputs = model(b_input_ids, b_attn_mask, b_features)
        loss = loss_fn(outputs, b_labels)
        total_train_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Loss de entrenamiento promedio: {avg_train_loss:.4f}")
    
    # Validación
    model.eval()
    total_val_loss = 0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch in val_dataloader:
            b_input_ids = batch[0].to(device)
            b_attn_mask = batch[1].to(device)
            b_features = batch[2].to(device)
            b_labels = batch[3].to(device)
            
            outputs = model(b_input_ids, b_attn_mask, b_features)
            loss = loss_fn(outputs, b_labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(b_labels.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_dataloader)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Loss de validación: {avg_val_loss:.4f} | Accuracy de validación: {val_acc:.4f}")
    
    # Guardar el mejor modelo basado en loss de validación
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model, PATH + "best_model_spanish_loss.pt")
        print("¡Modelo guardado como el mejor!")

print("\n--- Entrenamiento Completado. Reporte final del mejor modelo en Validación ---")
best_model = torch.load(PATH + "best_model_spanish_loss.pt")
best_model.eval()
final_preds = []
with torch.no_grad():
    for batch in val_dataloader:
        b_input_ids = batch[0].to(device)
        b_attn_mask = batch[1].to(device)
        b_features = batch[2].to(device)
        outputs = best_model(b_input_ids, b_attn_mask, b_features)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        final_preds.extend(preds)

print(classification_report(val_labels, final_preds))